# ⏰ EMN Doomsday Clock
## Equity Market Neutral Fund — Composite Risk Monitor

> **Physicist's framing:** Risk space is an N-dimensional manifold. Each indicator is a coordinate.
> The doomsday score is a scalar field computed as a weighted sum with non-linear interaction
> penalties for co-occurring stresses (analogous to resonance in coupled oscillators).

---

### Notebook structure

| Section | What it covers |
|---|---|
| 0 · Imports | Setup, colour palette, matplotlib theme |
| 1 · Data Model | Indicator universe, thresholds, category weights |
| 2 · Scoring Engine | Piecewise interpolation, aggregation, interaction boost |
| 2a · Curves | Visualise all 13 scoring functions |
| 3 · Dashboard | Gauge, category bars, radar, heatmap, decomposition |
| 4 · Scenarios | Baseline / Stressed / Crisis (2008) / Tail Risk |
| 5 · Sensitivity | OAT gradient — which indicators move the needle? |
| 6 · Monte Carlo | Score distribution under random market paths |
| 7 · Interactive | Live what-if sliders (requires ipywidgets) |
| 8 · History | 252-day OU simulation + rolling stats |
| 9 · Export | JSON snapshot for logging / alerting pipelines |
| 10 · Prod Notes | Data pipeline, alert thresholds, architecture |

**Dependencies:** `matplotlib`, `numpy`, `scipy` (optional), `ipywidgets` (optional).  
Core scoring logic requires only Python stdlib.

## 0 · Imports & Setup

In [ ]:
import math, random, json, datetime
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
from enum import Enum
from copy import deepcopy

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import warnings; warnings.filterwarnings('ignore')

# ── Terminal-green dark theme ──────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#050a08', 'axes.facecolor':  '#0a1210',
    'axes.edgecolor':   '#1a3028', 'axes.labelcolor': '#b0cfc0',
    'text.color':       '#b0cfc0', 'xtick.color':     '#3a6050',
    'ytick.color':      '#3a6050', 'grid.color':      '#1a3028',
    'grid.linewidth':   0.5,       'font.family':     'monospace',
    'figure.dpi':       120,
})

RISK_CMAP = LinearSegmentedColormap.from_list(
    'risk', ['#00ff88','#f0c040','#ff7820','#ff2244'], N=256)

def risk_color(score: float) -> str:
    t = max(0.0, min(1.0, score / 10.0))
    r, g, b, _ = RISK_CMAP(t)
    return f'#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}'

print("Setup complete. matplotlib", matplotlib.__version__)

## 1 · Data Model

Each `Indicator` carries:
- **value** — current reading (set by user or live feed)  
- **thresholds** — `[(val, score), ...]` breakpoints for piecewise-linear scoring  
- **weight** — fractional importance within its parent category

### Category weights

| Category | Weight | Rationale |
|---|---|---|
| Portfolio Health | 25% | Direct capital impact |
| Neutrality Check | 20% | Core mandate: market-neutral |
| Liquidity Risk | 20% | Unwind risk — often underestimated |
| Market Stress | 18% | Exogenous amplifier |
| Systemic Signals | 10% | Macro fragility / funding |
| Performance Decay | 7% | Leading indicator of model failure |

In [ ]:
class Regime(Enum):
    NOMINAL  = ("NOMINAL",  "#00ff88")
    WATCH    = ("WATCH",    "#f0c040")
    WARNING  = ("WARNING",  "#ff7820")
    DANGER   = ("DANGER",   "#ff2244")
    CRITICAL = ("CRITICAL", "#ff0000")
    def label(self): return self.value[0]
    def color(self): return self.value[1]


@dataclass
class Indicator:
    id: str; label: str; desc: str; unit: str
    value: float; default: float; lo: float; hi: float
    thresholds: List[Tuple[float,float]]; weight: float
    fmt: str = "{:.2f}"; raw_score: float = field(default=0.0, repr=False)

    def formatted_value(self):
        try:    return self.fmt.format(self.value)
        except: return str(self.value)

    def score_class(self):
        s = self.raw_score
        if s < 2.5: return "safe"
        if s < 4.5: return "watch"
        if s < 6.5: return "warning"
        return "danger"


@dataclass
class Category:
    id: str; label: str; weight: float
    indicators: List[Indicator]
    cat_score: float = field(default=0.0, repr=False)

    def score_class(self):
        s = self.cat_score
        if s < 2.5: return "safe"
        if s < 4.5: return "watch"
        if s < 6.5: return "warning"
        return "danger"


# Dangerous co-occurring pairs (id_a, id_b, multiplier, description)
# Physics: constructive interference between risk modes
INTERACTION_PAIRS = [
    ("beta",          "vix",          1.4, "Directional exposure in vol spike"),
    ("drawdown",      "leverage",     1.5, "Margin call cascade"),
    ("concentration", "bid_ask",      1.3, "Illiquid concentrated book"),
    ("drawdown",      "fund_flows",   1.4, "Redemption spiral"),
    ("bid_ask",       "fund_flows",   1.5, "Crowded unwind into illiquidity"),
    ("sovereign_cds", "leverage",     1.3, "Funding shock x leverage"),
    ("sharpe_decay",  "drawdown",     1.3, "Persistent alpha decay"),
    ("vol_of_vol",    "beta",         1.2, "Erratic P&L + directional drift"),
]


def build_categories() -> List[Category]:
    return [
        Category("portfolio_health", "Portfolio Health", 0.25, [
            Indicator("drawdown",  "Current Drawdown", "(Peak NAV - Current)/Peak",
                      "%", 8.0,8.0,0,35,[(0,0),(5,2),(10,5),(15,7),(20,9),(35,10)],0.6,"{:.1f}%"),
            Indicator("leverage",  "Leverage Ratio",   "Gross Exposure / Equity",
                      "x", 2.5,2.5,0,8, [(0,0),(2,1),(3,4),(4,6),(5,8),(8,10)],   0.4,"{:.2f}x"),
        ]),
        Category("neutrality", "Neutrality Check", 0.20, [
            Indicator("beta",       "Beta Exposure",   "Regression beta vs S&P 500",
                      "",  0.12,0.12,0,0.7,[(0,0),(0.1,1),(0.2,4),(0.3,7),(0.5,10)], 0.55,"{:+.3f}b"),
            Indicator("sector_net", "Max Sector Net",  "Max |long - short| per sector",
                      "%", 4.0,4.0,0,20, [(0,0),(3,1),(5,3),(8,5),(12,7),(20,10)],  0.45,"{:.1f}%"),
        ]),
        Category("liquidity", "Liquidity Risk", 0.20, [
            Indicator("bid_ask",       "Avg Bid-Ask Spread","Portfolio-weighted spread",
                      "%",0.3,0.3,0,2.5,[(0,0),(0.2,1),(0.5,4),(0.8,6),(1.2,8),(2.5,10)],0.5,"{:.3f}%"),
            Indicator("concentration", "HHI Concentration", "Herfindahl-Hirschman Index",
                      "",0.08,0.08,0,0.5,[(0,0),(0.05,1),(0.12,4),(0.2,6),(0.3,8),(0.5,10)],0.5,"{:.3f}"),
        ]),
        Category("market_stress", "Market Stress", 0.18, [
            Indicator("vix",        "VIX Index",       "CBOE 30-day implied vol",
                      "",18.0,18.0,8,90,[(8,0),(15,1),(20,3),(30,6),(45,8),(90,10)],0.45,"{:.1f}"),
            Indicator("fund_flows", "Fund Flows (mo.)","EMN category AUM change %",
                      "%",-2.0,-2.0,-25,5,[(-25,10),(-10,7),(-5,5),(-2,3),(0,1),(5,0)],0.30,"{:+.1f}%"),
            Indicator("ted_spread", "TED Spread",      "3M LIBOR - T-Bill (bps)",
                      "bps",20.0,20.0,0,250,[(0,0),(15,1),(30,3),(60,6),(100,8),(250,10)],0.25,"{:.0f}bps"),
        ]),
        Category("systemic", "Systemic Signals", 0.10, [
            Indicator("sovereign_cds","Sovereign CDS (US)","5Y CDS basis points",
                      "bps",30.0,30.0,0,250,[(0,0),(25,1),(50,4),(75,6),(100,8),(250,10)],0.5,"{:.0f}bps"),
            Indicator("gold_momentum","Gold 1M Momentum",  "Rolling 1-month return",
                      "%",1.5,1.5,-5,25,[(-5,0),(0,0),(5,3),(8,5),(12,7),(20,9),(25,10)],0.5,"{:+.1f}%"),
        ]),
        Category("performance_decay", "Performance Decay", 0.07, [
            Indicator("sharpe_decay","Sharpe Decay",     "30d Sharpe / 90d Sharpe",
                      "",0.85,0.85,0,1,[(1,0),(0.8,2),(0.6,5),(0.4,7),(0.2,9),(0,10)],0.6,"{:.2f}"),
            Indicator("vol_of_vol",  "Return Dispersion","Std(daily P&L) / mean|P&L|",
                      "",1.2,1.2,0,6,  [(0,0),(1,1),(1.5,3),(2.5,6),(3.5,8),(6,10)],   0.4,"{:.2f}"),
        ]),
    ]

cats_check = build_categories()
n_ind = sum(len(c.indicators) for c in cats_check)
print(f"Data model loaded: {n_ind} indicators across {len(cats_check)} categories")
for c in cats_check:
    print(f"  {c.label:<25} wt={c.weight:.0%}  ({len(c.indicators)} indicators)")

## 2 · Scoring Engine

### Pseudocode

```
score_indicator(value, thresholds):
    # Piecewise-linear — no cliff effects between bins
    if value <= t[0].v:  return t[0].s
    if value >= t[-1].v: return t[-1].s
    for (v0,s0),(v1,s1) in consecutive_pairs:
        if v0 <= value <= v1:
            return s0 + (value-v0)/(v1-v0) * (s1-s0)

category_score(indicators):
    return sum(ind.score * ind.weight) / sum(ind.weight)

interaction_boost(scores):
    # Resonance penalty: two dangerous modes >> their sum
    boost = 0
    for (a, b, mult, _) in INTERACTION_PAIRS:
        if scores[a] >= 6 AND scores[b] >= 6:
            boost += (scores[a]+scores[b])/2 * (mult-1)
    return min(boost, 2.5)     # cap preserves interpretability

composite(categories):
    base  = sum(category_score(cat) * cat.weight)
    boost = interaction_boost(all_scores)
    return min(base + boost, 10.0)
```

In [ ]:
def score_indicator(value: float, thresholds: List[Tuple[float,float]]) -> float:
    if not thresholds: return 0.0
    ascending = thresholds[-1][0] >= thresholds[0][0]
    if ascending:
        if value <= thresholds[0][0]:  return float(thresholds[0][1])
        if value >= thresholds[-1][0]: return float(thresholds[-1][1])
        for i in range(len(thresholds)-1):
            v0,s0 = thresholds[i]; v1,s1 = thresholds[i+1]
            if v0 <= value <= v1:
                return s0+(value-v0)/(v1-v0)*(s1-s0) if v1!=v0 else float(s0)
    else:
        if value >= thresholds[0][0]:  return float(thresholds[0][1])
        if value <= thresholds[-1][0]: return float(thresholds[-1][1])
        for i in range(len(thresholds)-1):
            v0,s0 = thresholds[i]; v1,s1 = thresholds[i+1]
            lo,hi = min(v0,v1),max(v0,v1)
            if lo <= value <= hi:
                return s0+(value-v0)/(v1-v0)*(s1-s0) if v1!=v0 else float(s0)
    return float(thresholds[-1][1])


def category_score(cat: Category) -> float:
    w = sum(i.weight for i in cat.indicators)
    return sum(i.raw_score*i.weight for i in cat.indicators)/w if w else 0.0


def interaction_boost(scores: Dict[str,float]) -> Tuple[float,List[str]]:
    boost, active = 0.0, []
    for id_a,id_b,mult,desc in INTERACTION_PAIRS:
        sa,sb = scores.get(id_a,0.0), scores.get(id_b,0.0)
        if sa >= 6.0 and sb >= 6.0:
            delta = (sa+sb)/2.0*(mult-1.0)
            boost += delta
            active.append(f"{desc}  (+{delta:.2f})")
    return min(boost, 2.5), active


def compute(categories: List[Category]) -> Tuple[float,float,List[str]]:
    for cat in categories:
        for ind in cat.indicators:
            ind.raw_score = score_indicator(ind.value, ind.thresholds)
        cat.cat_score = category_score(cat)
    all_scores = {ind.id: ind.raw_score for cat in categories for ind in cat.indicators}
    boost, boost_desc = interaction_boost(all_scores)
    base = sum(cat.cat_score*cat.weight for cat in categories)
    return min(base+boost, 10.0), boost, boost_desc


def classify(score: float) -> Regime:
    if score < 2.5: return Regime.NOMINAL
    if score < 4.5: return Regime.WATCH
    if score < 6.5: return Regime.WARNING
    if score < 8.5: return Regime.DANGER
    return Regime.CRITICAL


# ── Smoke test ─────────────────────────────────────────────────────────────
cats = build_categories()
s, b, d = compute(cats)
print(f"Baseline   composite={s:.3f}  boost={b:.3f}  [{classify(s).label()}]")

stress = {"drawdown":18,"leverage":4.5,"beta":0.28,"sector_net":9,
          "bid_ask":0.9,"concentration":0.22,"vix":42,"fund_flows":-8,
          "ted_spread":65,"sovereign_cds":70,"gold_momentum":7.5,
          "sharpe_decay":0.45,"vol_of_vol":2.8}
cats_s = build_categories()
for c in cats_s:
    for i in c.indicators:
        if i.id in stress: i.value = stress[i.id]
s2,b2,d2 = compute(cats_s)
print(f"Stressed   composite={s2:.3f}  boost={b2:.3f}  [{classify(s2).label()}]")
for x in d2: print(f"  . {x}")

## 2a · Indicator Scoring Curves

Each curve shows how a raw indicator value maps to a risk score [0,10] via piecewise-linear interpolation. The yellow dashed line marks the current (default) value.

Key insight: **no cliff effects** — a drawdown of 10.1% scores almost the same as 10.0%, unlike hard-threshold systems.

In [ ]:
cats_plot = build_categories()
all_inds_plot = [(cat.label, ind) for cat in cats_plot for ind in cat.indicators]
n = len(all_inds_plot)
ncols, nrows = 4, math.ceil(n/4)

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows*3.2))
fig.suptitle("Indicator Scoring Functions  (piecewise-linear interpolation)",
             fontsize=13, color='#b0cfc0', y=1.01)
axes_flat = axes.flatten()

for idx, (cat_label, ind) in enumerate(all_inds_plot):
    ax = axes_flat[idx]
    xs = np.linspace(ind.lo, ind.hi, 400)
    ys = [score_indicator(x, ind.thresholds) for x in xs]

    for xi, yi in zip(xs, ys):
        ax.axvline(xi, ymin=0, ymax=yi/10, color=risk_color(yi), alpha=0.07, linewidth=1)
    ax.plot(xs, ys, color='#b0cfc0', linewidth=1.8, zorder=3)

    tvx = [t[0] for t in ind.thresholds]
    tvy = [t[1] for t in ind.thresholds]
    ax.scatter(tvx, tvy, color='#00ff88', s=25, zorder=4, linewidths=0)

    ax.axvline(ind.value, color='#f0c040', linewidth=1.2, linestyle='--', alpha=0.8)
    cur_s = score_indicator(ind.value, ind.thresholds)
    ax.scatter([ind.value],[cur_s], color='#f0c040', s=50, zorder=5)

    for lo_s,hi_s,col in [(0,2.5,'#00ff88'),(2.5,4.5,'#f0c040'),
                            (4.5,6.5,'#ff7820'),(6.5,10,'#ff2244')]:
        ax.axhspan(lo_s,hi_s,alpha=0.04,color=col)

    ax.set_xlim(ind.lo,ind.hi); ax.set_ylim(0,10.5)
    ax.set_title(f"{ind.label}", fontsize=9, color='#e0f5ec', pad=3)
    ax.set_xlabel(cat_label, fontsize=7, color='#3a6050')
    ax.yaxis.set_tick_params(labelsize=7); ax.xaxis.set_tick_params(labelsize=7)
    ax.grid(True, alpha=0.3)

for i in range(n, len(axes_flat)):
    axes_flat[i].set_visible(False)

plt.tight_layout()
plt.savefig('scoring_curves.png', dpi=130, bbox_inches='tight', facecolor='#050a08')
plt.show()

## 3 · Main Dashboard

Single-figure composite dashboard: doomsday gauge, category bars, radar, indicator heatmap, score decomposition.

In [ ]:
def draw_dashboard(categories: List[Category], title: str = "Baseline"):
    composite, boost, boost_desc = compute(categories)
    regime = classify(composite)
    all_ind_map = {ind.id: ind for cat in categories for ind in cat.indicators}

    fig = plt.figure(figsize=(18,12))
    fig.patch.set_facecolor('#050a08')
    gs = gridspec.GridSpec(3,4,figure=fig,hspace=0.48,wspace=0.42)

    # 1. Gauge ───────────────────────────────────────────────────────────────
    ax_g = fig.add_subplot(gs[0,0], projection='polar')
    ax_g.set_facecolor('#0a1210')
    theta_all = np.linspace(np.pi*1.1,-np.pi*0.1,300)
    ax_g.plot(theta_all,[0.85]*300,color='#1a3028',linewidth=12,solid_capstyle='round')
    arc_len   = np.pi*1.2
    frac_f    = composite/10.0
    theta_f   = np.linspace(np.pi*1.1, np.pi*1.1-arc_len*frac_f, 200)
    for i in range(len(theta_f)-1):
        t_col = risk_color(composite*(i/(max(len(theta_f)-1,1))))
        ax_g.plot(theta_f[i:i+2],[0.85]*2,color=t_col,linewidth=12,solid_capstyle='butt')
    hand_ang = np.pi*1.1-arc_len*frac_f
    ax_g.annotate('',xy=(hand_ang,0.72),xytext=(0,0),
                  arrowprops=dict(arrowstyle='->',color='white',lw=1.5))
    ax_g.set_ylim(0,1); ax_g.set_rticks([]); ax_g.set_thetagrids([])
    ax_g.spines['polar'].set_visible(False)
    col_g = risk_color(composite)
    ax_g.text(0,0,f"{composite:.2f}",ha='center',va='center',
              fontsize=22,fontweight='bold',color=col_g,transform=ax_g.transData)
    ax_g.text(0,-0.22,"/ 10.00",ha='center',va='center',
              fontsize=9,color='#3a6050',transform=ax_g.transData)
    ax_g.set_title(f"{regime.label()}\n{title}",color=col_g,
                   fontsize=11,fontweight='bold',pad=4)

    # 2. Category bars ────────────────────────────────────────────────────────
    ax_c = fig.add_subplot(gs[0,1:3])
    cats_s = sorted(categories,key=lambda c:c.cat_score,reverse=True)
    ax_c.barh(range(len(cats_s)),[c.cat_score for c in cats_s],
              color=[risk_color(c.cat_score) for c in cats_s],alpha=0.85,height=0.55)
    ax_c.set_yticks(range(len(cats_s)))
    ax_c.set_yticklabels([c.label for c in cats_s],fontsize=9)
    ax_c.set_xlim(0,10.5)
    ax_c.axvline(composite,color='white',linestyle='--',linewidth=0.8,alpha=0.5)
    for i,(cat) in enumerate(cats_s):
        ax_c.text(cat.cat_score+0.15,i,f"{cat.cat_score:.1f}  wt={cat.weight:.0%}",
                  va='center',fontsize=8,color='#b0cfc0')
    ax_c.set_title("Category Scores",color='#b0cfc0',fontsize=10)
    for lo,hi,col in [(0,2.5,'#00ff88'),(2.5,4.5,'#f0c040'),
                       (4.5,6.5,'#ff7820'),(6.5,10,'#ff2244')]:
        ax_c.axvspan(lo,hi,alpha=0.04,color=col)

    # 3. Radar ────────────────────────────────────────────────────────────────
    RIDS = ["drawdown","leverage","beta","vix",
            "bid_ask","concentration","fund_flows","sovereign_cds"]
    RLBL = ["Drawdown","Leverage","Beta","VIX",
            "Bid-Ask","Conc.","Flows","CDS"]
    n_r   = len(RIDS)
    angs  = [2*np.pi*i/n_r for i in range(n_r)]+[0]
    ax_r  = fig.add_subplot(gs[0,3], projection='polar')
    ax_r.set_facecolor('#0a1210')
    ax_r.set_theta_zero_location('N')
    ax_r.set_theta_direction(-1)
    for ring in [2.5,5,7.5,10]:
        ax_r.plot(np.linspace(0,2*np.pi,100),[ring]*100,color='#1a3028',linewidth=0.5)
    rv = [all_ind_map[r].raw_score for r in RIDS]+[all_ind_map[RIDS[0]].raw_score]
    avg_sc = sum(all_ind_map[r].raw_score for r in RIDS)/n_r
    fc = risk_color(avg_sc)
    ax_r.plot(angs,rv,color=fc,linewidth=1.5)
    ax_r.fill(angs,rv,alpha=0.2,color=fc)
    ax_r.scatter(angs[:-1],rv[:-1],color=fc,s=20,zorder=3)
    ax_r.set_thetagrids(np.degrees(angs[:-1]),RLBL,fontsize=7)
    ax_r.tick_params(colors='#3a6050'); ax_r.set_rticks([]); ax_r.set_ylim(0,10)
    ax_r.set_title("Risk Radar",color='#b0cfc0',fontsize=10,pad=12)

    # 4. Heatmap ───────────────────────────────────────────────────────────────
    ax_h = fig.add_subplot(gs[1,:])
    all_il = [(ind.label,ind.raw_score,ind.formatted_value())
              for cat in categories for ind in cat.indicators]
    xpos = np.arange(len(all_il))
    ax_h.bar(xpos,[x[1] for x in all_il],
             color=[risk_color(x[1]) for x in all_il],alpha=0.85,width=0.65)
    ax_h.set_xticks(xpos)
    ax_h.set_xticklabels([x[0] for x in all_il],rotation=30,ha='right',fontsize=8)
    ax_h.set_ylim(0,11); ax_h.set_ylabel("Score",fontsize=9)
    ax_h.set_title("All Indicator Scores",color='#b0cfc0',fontsize=10)
    for lo,hi,col in [(0,2.5,'#00ff88'),(2.5,4.5,'#f0c040'),
                       (4.5,6.5,'#ff7820'),(6.5,10,'#ff2244')]:
        ax_h.axhspan(lo,hi,alpha=0.04,color=col)
    for i,(lbl,s,v) in enumerate(all_il):
        ax_h.text(i,s+0.15,f"{s:.1f}",ha='center',fontsize=7,color=risk_color(s))
        ax_h.text(i,-0.6,v,ha='center',fontsize=6.5,color='#3a6050')

    # 5. Decomposition ─────────────────────────────────────────────────────────
    ax_d = fig.add_subplot(gs[2,:2])
    contribs = sorted([(cat.label,cat.cat_score*cat.weight,cat.weight)
                        for cat in categories],key=lambda x:x[1],reverse=True)
    contribs.append(("Interaction\nBoost",boost,1))
    lbls  = [x[0] for x in contribs]
    vals  = [x[1] for x in contribs]
    cols_ = [risk_color(x[1]/x[2]) for x in contribs[:-1]]+['#ff7820']
    ax_d.bar(range(len(lbls)),vals,color=cols_,alpha=0.85)
    ax_d.axhline(sum(vals),color='white',linestyle='--',lw=0.8,
                 label=f"Total={composite:.2f}")
    ax_d.set_xticks(range(len(lbls)))
    ax_d.set_xticklabels(lbls,fontsize=8,rotation=15,ha='right')
    ax_d.set_title("Score Decomposition (weighted contributions)",color='#b0cfc0',fontsize=10)
    ax_d.legend(fontsize=8); ax_d.set_ylabel("Contribution",fontsize=9)

    # 6. Boost detail ──────────────────────────────────────────────────────────
    ax_b = fig.add_subplot(gs[2,2:])
    ax_b.axis('off')
    ax_b.text(0.05,0.95,"Interaction Boost Detail",fontsize=10,color='#b0cfc0',
              transform=ax_b.transAxes,va='top',fontweight='bold')
    ax_b.text(0.05,0.82,f"Total boost:  +{boost:.3f}",fontsize=10,
              color='#ff7820' if boost>0 else '#00ff88',
              transform=ax_b.transAxes,va='top')
    for k,desc in enumerate(boost_desc or ["No active interaction penalties"]):
        ax_b.text(0.05,0.69-k*0.14,f". {desc}",fontsize=8,
                  color='#ff7820',transform=ax_b.transAxes,va='top')
    ax_b.text(0.05,0.12,
              f"Composite = base({composite-boost:.3f}) + boost({boost:.3f}) = {composite:.3f}",
              fontsize=9,color='#e0f5ec',transform=ax_b.transAxes,va='bottom',fontweight='bold')

    fig.text(0.5,0.98,
             f"EMN DOOMSDAY CLOCK  |  {title}  |  "
             f"{datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')}",
             ha='center',fontsize=11,color='#00cc6a',fontweight='bold')
    plt.savefig(f'dashboard_{title.lower().replace(" ","_")}.png',
                dpi=130,bbox_inches='tight',facecolor='#050a08')
    plt.show()
    return composite, boost


cats_base = build_categories()
draw_dashboard(cats_base, "Baseline")

## 4 · Scenario Analysis

| Scenario | Analogy | Key drivers |
|---|---|---|
| **Baseline** | Normal market | Default values |
| **Stressed** | 2022 rate shock | Elevated VIX, drawdown, outflows |
| **Crisis (2008)** | GFC / LTCM | Margin calls, illiquidity, crowding |
| **Tail Risk** | Extinction-level | All indicators near maximum |

In [ ]:
SCENARIOS = {
    "Baseline":   {},
    "Stressed":   {"drawdown":12,"leverage":3.8,"beta":0.22,"sector_net":7,
                   "bid_ask":0.6,"concentration":0.15,"vix":28,"fund_flows":-4.5,
                   "ted_spread":40,"sovereign_cds":45,"gold_momentum":4.5,
                   "sharpe_decay":0.65,"vol_of_vol":2.0},
    "Crisis (2008)": {"drawdown":22,"leverage":5.5,"beta":0.38,"sector_net":14,
                      "bid_ask":1.4,"concentration":0.30,"vix":55,"fund_flows":-12,
                      "ted_spread":120,"sovereign_cds":95,"gold_momentum":11,
                      "sharpe_decay":0.30,"vol_of_vol":3.8},
    "Tail Risk":  {"drawdown":32,"leverage":7.0,"beta":0.55,"sector_net":18,
                   "bid_ask":2.0,"concentration":0.42,"vix":75,"fund_flows":-20,
                   "ted_spread":200,"sovereign_cds":180,"gold_momentum":18,
                   "sharpe_decay":0.10,"vol_of_vol":5.2},
}

def apply_scenario(vals: dict) -> List[Category]:
    cats = build_categories()
    for c in cats:
        for i in c.indicators:
            if i.id in vals: i.value = vals[i.id]
    return cats

results = {}
for name, vals in SCENARIOS.items():
    cats = apply_scenario(vals)
    comp, boost, bd = compute(cats)
    results[name] = {
        "composite": comp, "boost": boost, "regime": classify(comp).label(),
        "cat_scores": {cat.id: cat.cat_score for cat in cats},
        "ind_scores": {ind.id: ind.raw_score for cat in cats for ind in cat.indicators},
    }
    print(f"{name:20s}  composite={comp:.2f}  boost={boost:.2f}  [{classify(comp).label()}]")

In [ ]:
# ── Scenario comparison: composite bar + category heatmap ─────────────────
scen_names = list(results.keys())
composites = [results[n]["composite"] for n in scen_names]
boosts     = [results[n]["boost"]     for n in scen_names]
bases      = [c-b for c,b in zip(composites,boosts)]
cat_ids    = [c.id    for c in build_categories()]
cat_labels = [c.label for c in build_categories()]

fig, (ax1,ax2) = plt.subplots(1,2,figsize=(16,5))
fig.patch.set_facecolor('#050a08')

x = np.arange(len(scen_names))
ax1.bar(x,bases,  color=[risk_color(c) for c in composites],alpha=0.8,label='Base score')
ax1.bar(x,boosts,bottom=bases,color='#ff7820',alpha=0.7,label='Interaction boost')
ax1.set_xticks(x); ax1.set_xticklabels(scen_names,fontsize=10)
ax1.set_ylim(0,11)
for threshold,col in [(2.5,'#00ff88'),(4.5,'#f0c040'),(6.5,'#ff7820'),(8.5,'#ff2244')]:
    ax1.axhline(threshold,color=col,lw=0.6,ls='--',alpha=0.4)
for i,c in enumerate(composites):
    ax1.text(i,c+0.1,f"{c:.2f}",ha='center',fontsize=10,
             color=risk_color(c),fontweight='bold')
ax1.legend(fontsize=9)
ax1.set_title("Composite Score by Scenario",color='#b0cfc0',fontsize=11)
ax1.set_ylabel("Score")

mat = np.array([[results[n]["cat_scores"].get(cid,0) for cid in cat_ids]
                for n in scen_names])
im = ax2.imshow(mat,cmap=RISK_CMAP,vmin=0,vmax=10,aspect='auto')
ax2.set_xticks(range(len(cat_ids))); ax2.set_yticks(range(len(scen_names)))
ax2.set_xticklabels(cat_labels,rotation=30,ha='right',fontsize=8)
ax2.set_yticklabels(scen_names,fontsize=10)
for i in range(len(scen_names)):
    for j in range(len(cat_ids)):
        v = mat[i,j]
        ax2.text(j,i,f"{v:.1f}",ha='center',va='center',fontsize=9,
                 color='black' if v>5 else 'white',fontweight='bold')
plt.colorbar(im,ax=ax2,label='Score')
ax2.set_title("Category Scores Heatmap",color='#b0cfc0',fontsize=11)
plt.tight_layout()
plt.savefig('scenario_comparison.png',dpi=130,bbox_inches='tight',facecolor='#050a08')
plt.show()

## 5 · Sensitivity Analysis

**One-at-a-time (OAT) gradient** — sweep each indicator lo→hi, holding all others at default. Sensitivity = range of composite scores.

Physics: this is the partial derivative `∂composite/∂indicator` integrated over the full domain, i.e. the *total variation* each coordinate contributes to the scalar field.

In [ ]:
def sensitivity_analysis(base_cats, n_pts=80):
    sens = {}
    for cat_ref in base_cats:
        for ind_ref in cat_ref.indicators:
            xs, ys = [], []
            for v in np.linspace(ind_ref.lo, ind_ref.hi, n_pts):
                cats_c = deepcopy(base_cats)
                for cc in cats_c:
                    for ii in cc.indicators:
                        if ii.id == ind_ref.id: ii.value = v
                comp,_,_ = compute(cats_c)
                xs.append(v); ys.append(comp)
            sens[ind_ref.id] = (ind_ref.label, xs, ys, ind_ref.value)
    return sens

cats_base = build_categories()
sens = sensitivity_analysis(cats_base)
ranges = {k: (v[0], max(v[2])-min(v[2])) for k,v in sens.items()}
sorted_ranges = sorted(ranges.items(), key=lambda x: x[1][1], reverse=True)

print("Sensitivity ranking (composite range when swept full scale):")
for k,(lbl,rng) in sorted_ranges:
    bar = '|'*int(rng*8)
    print(f"  {lbl:<28}  delta={rng:.3f}  {bar}")

In [ ]:
fig, axes = plt.subplots(2,1,figsize=(14,10))
fig.patch.set_facecolor('#050a08')

# Bar chart
ax_rank = axes[0]
lbls_r = [v[0] for _,v in sorted_ranges]
rngs_r = [v[1] for _,v in sorted_ranges]
ax_rank.barh(range(len(lbls_r)),rngs_r,color=[risk_color(r) for r in rngs_r],alpha=0.85)
ax_rank.set_yticks(range(len(lbls_r))); ax_rank.set_yticklabels(lbls_r,fontsize=9)
ax_rank.set_xlabel("Composite Score Range (hi - lo) when indicator swept full scale",fontsize=9)
ax_rank.set_title("Sensitivity Ranking — OAT Analysis",color='#b0cfc0',fontsize=11)
ax_rank.axvline(1.0,color='#f0c040',ls='--',lw=0.8,alpha=0.6,label='>1.0 = significant')
ax_rank.legend(fontsize=8)

# Sweep curves for top-5
ax_sw = axes[1]
for k,(lbl,rng) in sorted_ranges[:5]:
    _,xs,ys,cv = sens[k]
    col = risk_color(rng)
    ax_sw.plot(np.linspace(0,1,len(xs)),ys,label=f"{lbl} (d{rng:.2f})",
               color=col,linewidth=1.5)
    ax_sw.scatter([np.interp(cv,[xs[0],xs[-1]],[0,1])],
                  [np.interp(cv,xs,ys)],color=col,s=50,zorder=4)

ax_sw.set_xlabel("Indicator value (normalised 0=lo, 1=hi)",fontsize=9)
ax_sw.set_ylabel("Composite Score",fontsize=9)
ax_sw.set_title("Top-5 Sensitive Indicators — OAT Sweep",color='#b0cfc0',fontsize=11)
ax_sw.legend(fontsize=8,loc='upper left')
for lo,hi,col in [(0,2.5,'#00ff88'),(2.5,4.5,'#f0c040'),
                   (4.5,6.5,'#ff7820'),(6.5,10,'#ff2244')]:
    ax_sw.axhspan(lo,hi,alpha=0.04,color=col)

plt.tight_layout()
plt.savefig('sensitivity.png',dpi=130,bbox_inches='tight',facecolor='#050a08')
plt.show()

## 6 · Monte Carlo Stress Testing

Draw `N=10,000` random states from `Normal(default, sigma)` where `sigma = 0.15 * range`, clipped to `[lo, hi]`. Compute composite for each draw.

This gives the **marginal distribution** of composite scores under parametric uncertainty — equivalent to integrating the risk scalar field over a Gaussian measure centred on the default state.

In [ ]:
def monte_carlo(n=10_000, sigma_frac=0.15, seed=42):
    rng      = np.random.default_rng(seed)
    cats_ref = build_categories()
    scores   = np.empty(n)
    for sim in range(n):
        cats_mc = deepcopy(cats_ref)
        for cat_mc in cats_mc:
            for ind_mc in cat_mc.indicators:
                sigma    = sigma_frac*(ind_mc.hi-ind_mc.lo)
                ind_mc.value = float(np.clip(rng.normal(ind_mc.default,sigma),
                                              ind_mc.lo,ind_mc.hi))
        scores[sim],_,_ = compute(cats_mc)
    return scores

print("Running 10,000 Monte Carlo simulations...")
mc_scores = monte_carlo(10_000)
print(f"  mean={mc_scores.mean():.3f}  std={mc_scores.std():.3f}")
print(f"  P05={np.percentile(mc_scores,5):.3f}  "
      f"P50={np.percentile(mc_scores,50):.3f}  "
      f"P95={np.percentile(mc_scores,95):.3f}  "
      f"P99={np.percentile(mc_scores,99):.3f}")
print()
for regime,lo,hi in [("NOMINAL",0,2.5),("WATCH",2.5,4.5),
                      ("WARNING",4.5,6.5),("DANGER",6.5,8.5),("CRITICAL",8.5,10)]:
    frac = ((mc_scores>=lo)&(mc_scores<hi)).mean()
    print(f"  {regime:<10}  {frac:.1%}  {'|'*int(frac*60)}")

In [ ]:
fig,(ax1,ax2) = plt.subplots(1,2,figsize=(16,5))
fig.patch.set_facecolor('#050a08')

ax1.hist(mc_scores,bins=80,density=True,color='#00ff88',alpha=0.5,edgecolor='none')
try:
    from scipy.stats import gaussian_kde
    kde = gaussian_kde(mc_scores)
    xs_k = np.linspace(0,10,300)
    ax1.plot(xs_k,kde(xs_k),color='white',linewidth=1.5)
except ImportError: pass

for lo,hi,col,lbl in [(0,2.5,'#00ff88','NOMINAL'),(2.5,4.5,'#f0c040','WATCH'),
                       (4.5,6.5,'#ff7820','WARNING'),(6.5,10,'#ff2244','DANGER+')]:
    ax1.axvspan(lo,hi,alpha=0.07,color=col)
    frac = ((mc_scores>=lo)&(mc_scores<hi)).mean()
    ax1.text((lo+hi)/2,0.02,f"{lbl}\n{frac:.1%}",ha='center',fontsize=8,
             color=col,transform=ax1.get_xaxis_transform())
ax1.axvline(mc_scores.mean(),color='white',ls='--',lw=1,label=f"mean={mc_scores.mean():.2f}")
ax1.axvline(np.percentile(mc_scores,95),color='#ff2244',ls='--',lw=1,
            label=f"P95={np.percentile(mc_scores,95):.2f}")
ax1.set_xlabel("Composite Score"); ax1.set_ylabel("Density")
ax1.set_title("Monte Carlo Distribution  (N=10,000)",color='#b0cfc0',fontsize=11)
ax1.legend(fontsize=9)

s_s = np.sort(mc_scores)
ax2.plot(s_s,np.arange(1,len(mc_scores)+1)/len(mc_scores),color='#00ff88',lw=1.5)
for lo,col in [(2.5,'#f0c040'),(4.5,'#ff7820'),(6.5,'#ff2244')]:
    p = (mc_scores<lo).mean()
    ax2.axvline(lo,color=col,ls='--',lw=0.8,alpha=0.6)
    ax2.axhline(p, color=col,ls='--',lw=0.8,alpha=0.6)
    ax2.text(lo+0.05,p-0.04,f"{p:.1%}<{lo}",fontsize=8,color=col)
ax2.set_xlabel("Composite Score"); ax2.set_ylabel("Cumulative Probability")
ax2.set_title("ECDF",color='#b0cfc0',fontsize=11); ax2.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig('montecarlo.png',dpi=130,bbox_inches='tight',facecolor='#050a08')
plt.show()

## 7 · Interactive What-If Explorer

Requires `ipywidgets`. Install with: `pip install ipywidgets`

All 13 sliders update the dashboard live.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not available. Install: pip install ipywidgets")

if HAS_WIDGETS:
    cats_live = build_categories()
    sliders   = {}
    for cat in cats_live:
        for ind in cat.indicators:
            sliders[ind.id] = widgets.FloatSlider(
                value=ind.default, min=ind.lo, max=ind.hi,
                step=(ind.hi-ind.lo)/200,
                description=ind.label[:20],
                style={'description_width':'170px'},
                layout=widgets.Layout(width='520px'),
                continuous_update=False,
            )

    out = widgets.Output()

    def on_change(**kwargs):
        for cat in cats_live:
            for ind in cat.indicators: ind.value = kwargs[ind.id]
        comp,boost,boost_desc = compute(cats_live)
        regime = classify(comp)
        col_g  = risk_color(comp)

        with out:
            clear_output(wait=True)
            fig,axes = plt.subplots(1,3,figsize=(16,4),
                                    gridspec_kw={'width_ratios':[1,1.2,1.8]})
            fig.patch.set_facecolor('#050a08')

            # Gauge
            ax_g = plt.subplot(1,3,1,projection='polar')
            ax_g.set_facecolor('#0a1210')
            tg = np.linspace(np.pi*1.1,-np.pi*0.1,300)
            ax_g.plot(tg,[0.85]*300,color='#1a3028',lw=12,solid_capstyle='round')
            frac = comp/10.0; arc = np.pi*1.2
            tf = np.linspace(np.pi*1.1, np.pi*1.1-arc*frac, 200)
            for i in range(len(tf)-1):
                ax_g.plot(tf[i:i+2],[0.85]*2,color=col_g,lw=12,solid_capstyle='butt')
            ha = np.pi*1.1-arc*frac
            ax_g.annotate('',xy=(ha,0.72),xytext=(0,0),
                           arrowprops=dict(arrowstyle='->',color='white',lw=2))
            ax_g.set_ylim(0,1); ax_g.set_rticks([]); ax_g.set_thetagrids([])
            ax_g.spines['polar'].set_visible(False)
            ax_g.text(0,0,f"{comp:.2f}",ha='center',va='center',
                      fontsize=20,fontweight='bold',color=col_g,transform=ax_g.transData)
            ax_g.text(0,-0.22,"/ 10",ha='center',va='center',
                      fontsize=9,color='#3a6050',transform=ax_g.transData)
            ax_g.set_title(f"{regime.label()}  +{boost:.2f}",
                           color=col_g,fontsize=10,fontweight='bold')

            # Radar
            RIDS = ["drawdown","leverage","beta","vix",
                    "bid_ask","concentration","fund_flows","sovereign_cds"]
            RLBL = ["Drw","Lev","Bet","VIX","B/A","Con","Flw","CDS"]
            nr = len(RIDS); angs = [2*np.pi*i/nr for i in range(nr)]+[0]
            ax_r = plt.subplot(1,3,2,projection='polar')
            ax_r.set_facecolor('#0a1210')
            ax_r.set_theta_zero_location('N'); ax_r.set_theta_direction(-1)
            for ring in [2.5,5,7.5,10]:
                ax_r.plot(np.linspace(0,2*np.pi,100),[ring]*100,color='#1a3028',lw=0.4)
            im_ = {ind.id:ind for c in cats_live for ind in c.indicators}
            rv  = [im_[r].raw_score for r in RIDS]+[im_[RIDS[0]].raw_score]
            ax_r.plot(angs,rv,color=col_g,lw=1.5)
            ax_r.fill(angs,rv,alpha=0.2,color=col_g)
            ax_r.set_thetagrids(np.degrees(angs[:-1]),RLBL,fontsize=7,color='#3a6050')
            ax_r.set_ylim(0,10); ax_r.set_rticks([])
            ax_r.set_title("Radar",color='#b0cfc0',fontsize=9)

            # Category bars
            ax_cb = plt.subplot(1,3,3)
            ax_cb.set_facecolor('#0a1210')
            cat_nm = [c.label for c in cats_live]
            cat_sc = [c.cat_score for c in cats_live]
            ax_cb.barh(range(len(cats_live)),cat_sc,
                       color=[risk_color(s) for s in cat_sc],alpha=0.85)
            ax_cb.set_yticks(range(len(cats_live)))
            ax_cb.set_yticklabels(cat_nm,fontsize=8); ax_cb.set_xlim(0,10.5)
            for i,s in enumerate(cat_sc):
                ax_cb.text(s+0.1,i,f"{s:.1f}",va='center',fontsize=8,color=risk_color(s))
            ax_cb.set_title("Category Scores",color='#b0cfc0',fontsize=9)
            plt.tight_layout(); plt.show()

    # Slider grid (3 columns)
    groups = []
    for cat in cats_live:
        groups.append(widgets.VBox(
            [widgets.HTML(f"<b style='color:#00cc6a'>{cat.label}</b>")] +
            [sliders[ind.id] for ind in cat.indicators]))

    grid = widgets.GridBox(groups,
                           layout=widgets.Layout(grid_template_columns="repeat(3,1fr)"))
    iout = widgets.interactive_output(
        on_change,{ind.id:sliders[ind.id] for c in cats_live for ind in c.indicators})
    display(grid, out)
    on_change(**{ind.id:ind.value for c in cats_live for ind in c.indicators})

## 8 · Score History — Time-Series Simulation

252-day (1 trading year) simulation using Ornstein-Uhlenbeck mean-reverting processes:

$$dX_t = \theta(\mu - X_t)\,dt + \sigma\,dW_t$$

θ = mean-reversion speed, μ = default value (long-run mean), σ = daily volatility.

In [ ]:
def simulate_history(n_days=252, seed=0, crisis_day=None):
    rng  = np.random.default_rng(seed)
    cats = build_categories()
    OU   = {"vix":(0.15,0.04),"bid_ask":(0.10,0.03),"gold_momentum":(0.12,0.035),
            "ted_spread":(0.08,0.025),"fund_flows":(0.10,0.03),
            "sovereign_cds":(0.06,0.02),"drawdown":(0.05,0.015),
            "leverage":(0.08,0.015),"beta":(0.12,0.02),"sector_net":(0.10,0.025),
            "concentration":(0.06,0.015),"sharpe_decay":(0.08,0.02),
            "vol_of_vol":(0.10,0.025)}
    history = []
    for day in range(n_days):
        if crisis_day and day == crisis_day:
            for c in cats:
                for i in c.indicators: i.value += 0.6*(i.hi-i.value)
        for c in cats:
            for i in c.indicators:
                theta,sfrac = OU.get(i.id,(0.10,0.02))
                sigma = sfrac*(i.hi-i.lo)
                i.value = float(np.clip(i.value+theta*(i.default-i.value)
                                        +rng.normal(0,sigma), i.lo, i.hi))
        comp,boost,_ = compute(cats)
        history.append({"day":day,"composite":comp,"boost":boost,
                         "regime":classify(comp).label(),
                         "cat_scores":{c.id:round(c.cat_score,3) for c in cats}})
    return history

print("Simulating histories...")
hist_n = simulate_history(252,seed=7)
hist_c = simulate_history(252,seed=7,crisis_day=120)
scores_n = [h["composite"] for h in hist_n]
scores_c = [h["composite"] for h in hist_c]
print(f"Normal  mean={np.mean(scores_n):.2f}  max={max(scores_n):.2f}")
print(f"Crisis  mean={np.mean(scores_c):.2f}  max={max(scores_c):.2f}  "
      f"danger_days={sum(1 for s in scores_c if s>=6.5)}")

In [ ]:
fig,axes = plt.subplots(3,1,figsize=(16,12),sharex=True)
fig.patch.set_facecolor('#050a08')
days = range(252)

ax1 = axes[0]
for i in range(len(scores_n)-1):
    ax1.fill_between([i,i+1],[0]*2,[scores_n[i],scores_n[i+1]],
                     color=risk_color(scores_n[i]),alpha=0.25)
ax1.plot(days,scores_n,color='#00ff88',lw=1.2,label='Normal')
ax1.plot(days,scores_c,color='#ff2244',lw=1.0,alpha=0.7,ls='--',label='Crisis (day 120)')
ax1.axvline(120,color='#ff2244',lw=0.8,ls=':',alpha=0.6)
for lo,hi,col in [(0,2.5,'#00ff88'),(2.5,4.5,'#f0c040'),
                   (4.5,6.5,'#ff7820'),(6.5,10,'#ff2244')]:
    ax1.axhspan(lo,hi,alpha=0.04,color=col)
ax1.set_ylim(0,10); ax1.set_ylabel("Composite Score")
ax1.set_title("252-Day Score History Simulation",color='#b0cfc0',fontsize=11)
ax1.legend(fontsize=9,loc='upper right')

cat_ids_h = list(hist_n[0]["cat_scores"].keys())
cat_wts   = {c.id:c.weight for c in build_categories()}
cat_mat   = np.array([[h["cat_scores"].get(cid,0)*cat_wts.get(cid,0)
                        for cid in cat_ids_h] for h in hist_n])
CCOLORS = ['#00ff88','#40d0a0','#f0c040','#ff7820','#ff4466','#cc44ff']
ax2 = axes[1]
bottom = np.zeros(252)
for j,cid in enumerate(cat_ids_h):
    ax2.fill_between(days,bottom,bottom+cat_mat[:,j],
                     color=CCOLORS[j%len(CCOLORS)],alpha=0.6,
                     label=build_categories()[j].label)
    bottom += cat_mat[:,j]
ax2.set_ylabel("Weighted Contribution")
ax2.set_title("Category Contributions Over Time",color='#b0cfc0',fontsize=10)
ax2.legend(fontsize=7,loc='upper right',ncol=3)

W = 20; arr = np.array(scores_n)
rm = np.convolve(arr,np.ones(W)/W,mode='valid')
rs = np.array([arr[i:i+W].std() for i in range(len(arr)-W+1)])
xr = range(W-1,252)
ax3 = axes[2]
ax3.plot(days,scores_n,color='#3a6050',lw=0.8,alpha=0.5)
ax3.plot(xr,rm,color='#00ff88',lw=1.5,label=f'{W}d rolling mean')
ax3.fill_between(xr,rm-rs,rm+rs,color='#00ff88',alpha=0.15,label='+-1sigma')
ax3.set_xlabel("Trading Day"); ax3.set_ylabel("Composite Score")
ax3.set_title(f"Rolling {W}-Day Mean +/- sigma",color='#b0cfc0',fontsize=10)
ax3.legend(fontsize=9)
plt.tight_layout()
plt.savefig('score_history.png',dpi=130,bbox_inches='tight',facecolor='#050a08')
plt.show()

## 9 · JSON Snapshot Export

Serialises full risk state — suitable for logging to InfluxDB/DuckDB, triggering PagerDuty alerts, or feeding a Streamlit/Grafana dashboard.

In [ ]:
def export_snapshot(categories, filename=None):
    composite,boost,boost_desc = compute(categories)
    ts   = datetime.datetime.utcnow().isoformat()+"Z"
    RECS = {"NOMINAL":"Business as usual. Standard monitoring.",
            "WATCH":  "Increase monitoring cadence. Review factor exposures.",
            "WARNING":"Risk reduction warranted. Trim gross exposure.",
            "DANGER": "De-gross aggressively. Halt new trades.",
            "CRITICAL":"Wind-down protocol. Notify prime broker."}
    snap = {
        "metadata": {"timestamp":ts,"model_version":"2.0","generator":"EMN Doomsday Clock"},
        "summary":  {"composite_score":round(composite,4),
                     "regime":classify(composite).label(),
                     "interaction_boost":round(boost,4),
                     "boost_active":boost_desc,
                     "recommendation":RECS[classify(composite).label()]},
        "categories":[{
            "id":cat.id,"label":cat.label,"weight":cat.weight,
            "score":round(cat.cat_score,4),"regime":cat.score_class(),
            "indicators":[{"id":ind.id,"label":ind.label,"value":round(ind.value,6),
                           "raw_score":round(ind.raw_score,4),"weight":ind.weight,
                           "regime":ind.score_class(),"formatted":ind.formatted_value()}
                          for ind in cat.indicators]}
            for cat in categories]}
    if filename:
        with open(filename,'w') as f: json.dump(snap,f,indent=2)
        print(f"Saved -> {filename}")
    return snap

cats_exp = build_categories()
snap = export_snapshot(cats_exp,"emn_baseline_snapshot.json")
print(json.dumps(snap["summary"],indent=2))

cats_cr = apply_scenario(SCENARIOS["Crisis (2008)"])
snap_cr = export_snapshot(cats_cr,"emn_crisis_snapshot.json")
print()
print(json.dumps(snap_cr["summary"],indent=2))

## 10 · Production Integration Notes

### Data pipeline

```
intraday (every 15 min):
  vix           <- CBOE via yfinance / Bloomberg CBOE VIX
  bid_ask       <- Polygon.io real-time NBBO quotes
  gold_momentum <- Continuous front-month futures
  ted_spread    <- FRED: TB3MS (T-Bill) vs ICE LIBOR 3M
  sovereign_cds <- Bloomberg CDSW or ICE CDS feed

daily (end of day, after NAV strike):
  drawdown      <- Fund accounting NAV history
  leverage      <- Prime broker gross exposure report
  beta          <- Rolling 60d OLS regression vs SPX returns
  sector_net    <- Position file x GICS sector classification
  concentration <- Herfindahl-Hirschman from position weights
  fund_flows    <- Morningstar Direct API or HFRI monthly
  sharpe_decay  <- Computed from internal P&L time series
  vol_of_vol    <- Computed from internal P&L time series
```

### Alert matrix

| Condition | Action |
|---|---|
| Composite >= 4.5 (WATCH) | Email to PM + risk team |
| Composite >= 6.5 (DANGER) | Immediate page, freeze new trades |
| Composite >= 8.5 (CRITICAL) | Wind-down protocol, prime broker call |
| 1-day delta >= 1.5 | Immediate review regardless of absolute level |
| Any single indicator >= 8.0 | Sub-system alert to relevant desk |
| Interaction boost > 1.0 | Regime correlation warning to PM |

### Recommended architecture

```
[Data Feeds]
    |
    v
[Normalisation Layer]  <- validate ranges, handle stale data
    |
    v
[Scoring Engine]       <- this notebook, deployed as Lambda / Docker
    |
    +-> [Time-Series DB]     InfluxDB / DuckDB / TimescaleDB
    +-> [Alert Engine]       PagerDuty / Slack webhook
    +-> [Dashboard]          Grafana / Streamlit
```

### Extending the model

To add a new indicator, define it in `build_categories()`:
```python
Indicator("factor_crowding", "Factor Crowding", "13F consensus overlap",
          "%", 0.3, 0.3, 0, 1,
          [(0,0),(0.3,2),(0.5,5),(0.7,8),(1.0,10)], 0.5, "{:.2f}")
```
Then re-run the sensitivity analysis to verify it doesn't dominate unexpectedly,
and add any relevant interaction pairs to `INTERACTION_PAIRS`.